# LLM Evaluation Metrics Analysis

This notebook automates the process of importing, aggregating, and statistically analyzing evaluation metrics obtained from LLM testing questionnaires.

### Extracted Parameters:
* **Dataset & Model:** Derived from the hierarchical folder structure.
* **Configurations:** Extracted directly from the filename (`reasoning`, `personas`, `shot`), representing the specific experimental variables.

### Library Imports and Path Definition

In [1]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

# Define the provided base directory path
metrics_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/metrics")

# Pandas display configuration for optimal readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### File and Directory Structure Parsing Function

In [2]:
def parse_experiment_file(file_path):
    """
    Analyzes the file path to extract dataset, model, and experimental parameters.
    Handles both boolean reasoning (true/false) and leveled reasoning (low/medium/high).
    """
    stem = file_path.stem
    
    # Updated pattern: reasoning can now be true, false, low, medium, or high
    pattern = r"reasoning_(true|false|low|medium|high)_personas_(true|false)_(zero-shot|few-shot)"
    match = re.match(pattern, stem)
    
    if not match:
        print(f"[PARSING ERROR] Filename '{stem}' does not match the updated regex pattern.")
        return None

    # Keep reasoning as a string to preserve 'low', 'medium', 'high', 'true', or 'false'
    reasoning = match.group(1)
    personas = match.group(2) == 'true'
    shot = match.group(3)

    # Extract model and dataset from directory structure
    model = file_path.parent.name
    dataset = file_path.parent.parent.name
    
    return {
        "dataset": dataset,
        "model": model,
        "personas": personas,
        "reasoning": reasoning,
        "shot": shot
    }

### Recursive Loading and Run Aggregation

In [3]:
def load_and_aggregate_all(base_path):
    all_configs_data = []
    
    # Trova tutti i file .json ricorsivamente
    file_list = list(base_path.rglob("*.json"))
    
    print(f"[INFO] Found a total of {len(file_list)} JSON files in the directory.")
    
    if not file_list:
        print(f"Warning: No JSON files found at all in path: {base_path}")
        return pd.DataFrame()
        
    for file_path in file_list:
        metadata = parse_experiment_file(file_path)
        
        if metadata is None:
            continue
            
        with open(file_path, 'r', encoding='utf-8') as f:
            runs = json.load(f)
        
        mean_accuracies = []
        mean_logprobs = []
        subject_accuracies = {}
        
        # Accesso diretto e rigido alle chiavi certificate del file
        for run in runs:
            mean_accuracies.append(run['accuracy']['mean_accuracy'])
            mean_logprobs.append(run['confidence_score']['mean_extracted_logprob'])
            
            acc_per_subj = run['accuracy']['accuracy_per_subject']
            for subj, acc_val in acc_per_subj.items():
                if subj not in subject_accuracies:
                    subject_accuracies[subj] = []
                subject_accuracies[subj].append(acc_val)
                
        # Logica di mappatura: 'medium' viene impostato a None per essere escluso dall'inter-modello
        reasoning_raw = metadata["reasoning"]
        if reasoning_raw == 'low':
            reasoning_mapped = 'false'
        elif reasoning_raw == 'high':
            reasoning_mapped = 'true'
        elif reasoning_raw == 'medium':
            reasoning_mapped = None  # Pandas lo escluderà automaticamente dalle tabelle pivot inter-modello
        else:
            reasoning_mapped = reasoning_raw
            
        current_dataset = metadata["dataset"]
            
        aggregated_row = {
            "dataset": current_dataset,
            "model": metadata["model"],
            "personas": metadata["personas"],
            "reasoning": reasoning_raw,          
            "reasoning_mapped": reasoning_mapped, 
            "shot": metadata["shot"],
            "avg_mean_accuracy": np.mean(mean_accuracies),
            "avg_confidence_logprob": np.mean(mean_logprobs)
        }
        
        # I subject vengono tracciati includendo il nome del dataset per evitare collisioni nell'analisi globale
        for subj, acc_list in subject_accuracies.items():
            aggregated_row[f"avg_acc_{current_dataset}_{subj}"] = np.mean(acc_list)
            
        all_configs_data.append(aggregated_row)
        
    print(f"[SUCCESS] Successfully processed {len(all_configs_data)} configuration files.")
    return pd.DataFrame(all_configs_data)

# Esecuzione della pipeline
df_metrics = load_and_aggregate_all(metrics_base)
df_metrics.head()

[INFO] Found a total of 176 JSON files in the directory.
[SUCCESS] Successfully processed 176 configuration files.


,dataset,model,personas,reasoning,reasoning_mapped,shot,avg_mean_accuracy,avg_confidence_logprob,avg_acc_aime_math,avg_acc_mmlu_abstract_algebra,avg_acc_mmlu_anatomy,avg_acc_mmlu_astronomy,avg_acc_mmlu_business_ethics,avg_acc_mmlu_clinical_knowledge,avg_acc_mmlu_college_biology,avg_acc_mmlu_college_chemistry,avg_acc_mmlu_college_computer_science,avg_acc_mmlu_college_mathematics,avg_acc_mmlu_college_medicine,avg_acc_mmlu_college_physics,avg_acc_mmlu_computer_security,avg_acc_mmlu_conceptual_physics,avg_acc_mmlu_econometrics,avg_acc_mmlu_electrical_engineering,avg_acc_mmlu_elementary_mathematics,avg_acc_mmlu_formal_logic,avg_acc_mmlu_global_facts,avg_acc_mmlu_high_school_biology,avg_acc_mmlu_high_school_chemistry,avg_acc_mmlu_high_school_computer_science,avg_acc_mmlu_high_school_european_history,avg_acc_mmlu_high_school_geography,avg_acc_mmlu_high_school_government_and_politics,avg_acc_mmlu_high_school_macroeconomics,avg_acc_mmlu_high_school_mathematics,avg_acc_mmlu_high_school_microeconomics,avg_acc_mmlu_high_school_physics,avg_acc_mmlu_high_school_psychology,avg_acc_mmlu_high_school_statistics,avg_acc_mmlu_high_school_us_history,avg_acc_mmlu_high_school_world_history,avg_acc_mmlu_human_aging,avg_acc_mmlu_human_sexuality,avg_acc_mmlu_international_law,avg_acc_mmlu_jurisprudence,avg_acc_mmlu_logical_fallacies,avg_acc_mmlu_machine_learning,avg_acc_mmlu_management,avg_acc_mmlu_marketing,avg_acc_mmlu_medical_genetics,avg_acc_mmlu_miscellaneous,avg_acc_mmlu_moral_disputes,avg_acc_mmlu_moral_scenarios,avg_acc_mmlu_nutrition,avg_acc_mmlu_philosophy,avg_acc_mmlu_prehistory,avg_acc_mmlu_professional_accounting,avg_acc_mmlu_professional_law,avg_acc_mmlu_professional_medicine,avg_acc_mmlu_professional_psychology,avg_acc_mmlu_public_relations,avg_acc_mmlu_security_studies,avg_acc_mmlu_sociology,avg_acc_mmlu_us_foreign_policy,avg_acc_mmlu_virology,avg_acc_mmlu_world_religions,avg_acc_mmlu_pro_math,avg_acc_mmlu_pro_health,avg_acc_mmlu_pro_physics,avg_acc_mmlu_pro_business,avg_acc_mmlu_pro_biology,avg_acc_mmlu_pro_chemistry,avg_acc_mmlu_pro_computer science,avg_acc_mmlu_pro_economics,avg_acc_mmlu_pro_engineering,avg_acc_mmlu_pro_philosophy,avg_acc_mmlu_pro_other,avg_acc_mmlu_pro_history,avg_acc_mmlu_pro_psychology,avg_acc_mmlu_pro_law,avg_acc_gpqa_Biology,avg_acc_gpqa_Physics,avg_acc_gpqa_Chemistry
0,aime,openai_gpt-oss-20b,True,high,true,few-shot,0.906667,-0.107745,0.906667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,aime,openai_gpt-oss-20b,False,low,false,few-shot,0.980000,-0.145372,0.980000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,aime,openai_gpt-oss-20b,False,low,false,zero-shot,0.973333,-0.183704,0.973333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,aime,openai_gpt-oss-20b,False,medium,NaN,few-shot,0.980000,-0.004769,0.980000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aime,openai_gpt-oss-20b,True,medium,NaN,zero-shot,0.980000,-0.014569,0.980000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

## 1. Inter-Model Analysis

This section provides a direct side-by-side comparison of different models evaluated under identical experimental conditions, highlighting variations in general accuracy, subject-level accuracy, and response confidence.

### Generating Inter-Model Reports (Per-Dataset and Global)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

if not df_metrics.empty:
    images_base_path = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/images")
    inter_images_path = images_base_path / "inter_model"
    inter_images_path.mkdir(parents=True, exist_ok=True)
    
    df_inter = df_metrics[df_metrics['reasoning_mapped'].notna()].copy()
    sns.set_theme(style="whitegrid")
    
    personas_palette = {False: "#7f8c8d", True: "#e67e22"}
    
    print("==================================================")
    print("### 1.1 PER-DATASET INTER-MODEL ANALYSIS & PLOTTING")
    print("==================================================")
    
    for dataset_name, df_dataset in df_inter.groupby('dataset'):
        print(f"\nProcessing Plots for Dataset: {dataset_name.upper()}")
        dataset_img_path = inter_images_path / "per_dataset" / dataset_name
        dataset_img_path.mkdir(parents=True, exist_ok=True)
        
        # Creiamo una mappatura fissa dei modelli in indici numerici (1, 2, 3...) per questo dataset
        unique_models_ds = sorted(df_dataset['model'].unique())
        model_to_idx = {model: f"Model {i+1}" for i, model in enumerate(unique_models_ds)}
        df_dataset['model_idx'] = df_dataset['model'].map(model_to_idx)
        
        # Testo della legenda che associa il numero al nome reale del modello
        model_legend_text = "\n".join([f"Model {i+1}: {model}" for i, model in enumerate(unique_models_ds)])
        
        for (reasoning_val, shot_val), df_config in df_dataset.groupby(['reasoning_mapped', 'shot']):
            
            # Applichiamo la mappatura anche al subset di configurazione corrente
            df_config = df_config.copy()
            df_config['model_idx'] = df_config['model'].map(model_to_idx)
            df_config = df_config.sort_values('model_idx')
            
            # ----------------------------------------------------
            # GRAFICO 1: ACCURACY
            # ----------------------------------------------------
            fig, ax = plt.subplots(figsize=(10, 6))
            
            barplot_acc = sns.barplot(
                data=df_config,
                x='model_idx',
                y='avg_mean_accuracy',
                hue='personas',
                ax=ax,
                palette=personas_palette
            )
            
            ax.set_title(f"Dataset: {dataset_name} | Reasoning: {reasoning_val} | Shot: {shot_val}\nEffect of Personas on Accuracy")
            ax.set_ylabel("Accuracy")
            ax.set_xlabel("Models (See Legend)")
            ax.set_ylim(0, 1.1)
            
            # Creazione delle legende separate: Personas (grafica) e Models (testuale)
            handles, labels = ax.get_legend_handles_labels()
            leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.add_artist(leg_personas)
            
            # Aggiungiamo la legenda testuale dei modelli sotto quella delle personas
            ax.text(1.05, 0.7, f"Model Index Key:\n{model_legend_text}", transform=ax.transAxes, 
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                    fontsize=9, verticalalignment='top')
            
            for container in ax.containers:
                ax.bar_label(container, fmt='%.3f', padding=3, fontweight='bold', fontsize=9)
                
            plt.tight_layout()
            filename_acc = f"accuracy_reasoning_{reasoning_val}_{shot_val}.png"
            fig.savefig(dataset_img_path / filename_acc, dpi=300, bbox_inches='tight')
            # plt.show()
            plt.close()
            
            # ----------------------------------------------------
            # GRAFICO 2: CONFIDENCE LOGPROB
            # ----------------------------------------------------
            fig, ax = plt.subplots(figsize=(10, 6))
            
            df_config_inverted = df_config.copy()
            df_config_inverted['abs_confidence_logprob'] = df_config_inverted['avg_confidence_logprob'].abs()
            
            barplot_conf = sns.barplot(
                data=df_config_inverted,
                x='model_idx',
                y='abs_confidence_logprob',
                hue='personas',
                ax=ax,
                palette=personas_palette
            )
            
            ticks = ax.get_yticks()
            ax.set_yticks(ticks)
            ax.set_yticklabels([f"-{t:.2f}" if t != 0 else "0.00" for t in ticks])
            
            ax.set_title(f"Dataset: {dataset_name} | Reasoning: {reasoning_val} | Shot: {shot_val}\nEffect of Personas on Extracted Confidence Score")
            ax.set_ylabel("Confidence Logprob")
            ax.set_xlabel("Models (See Legend)")
            
            handles, labels = ax.get_legend_handles_labels()
            leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.add_artist(leg_personas)
            
            ax.text(1.05, 0.7, f"Model Index Key:\n{model_legend_text}", transform=ax.transAxes, 
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                    fontsize=9, verticalalignment='top')
            
            for p in barplot_conf.patches:
                height = p.get_height()
                if not np.isnan(height) and height > 0:
                    original_val = -height
                    ax.annotate(f'{original_val:.4f}',
                                (p.get_x() + p.get_width() / 2., height),
                                ha='center', va='bottom',
                                xytext=(0, 3), textcoords='offset points', fontweight='bold', fontsize=8)
            
            plt.tight_layout()
            filename_conf = f"confidence_reasoning_{reasoning_val}_{shot_val}.png"
            fig.savefig(dataset_img_path / filename_conf, dpi=300, bbox_inches='tight')
            # plt.show()
            plt.close()

    print("\n" + "="*50 + "\n")
    print("### 1.2 GLOBAL INTER-MODEL EVALUATION (DATASETS IGNORED)")
    print("==================================================")
    
    global_img_path = inter_images_path / "global"
    global_img_path.mkdir(parents=True, exist_ok=True)
    
    unique_models_global = sorted(df_inter['model'].unique())
    global_model_to_idx = {model: f"Model {i+1}" for i, model in enumerate(unique_models_global)}
    global_legend_text = "\n".join([f"Model {i+1}: {model}" for i, model in enumerate(unique_models_global)])
    
    for (reasoning_val, shot_val), df_global_config in df_inter.groupby(['reasoning_mapped', 'shot']):
        
        df_global_config = df_global_config.copy()
        df_global_config['model_idx'] = df_global_config['model'].map(global_model_to_idx)
        df_global_config = df_global_config.sort_values('model_idx')
        
        # Grafico Globale Accuracy
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(
            data=df_global_config,
            x='model_idx',
            y='avg_mean_accuracy',
            hue='personas',
            errorbar=None,
            ax=ax,
            palette=personas_palette
        )
        ax.set_title(f"GLOBAL ANALYSIS | Reasoning: {reasoning_val} | Shot: {shot_val}\nEffect of Personas on Accuracy")
        ax.set_ylabel("Accuracy")
        ax.set_xlabel("Models (See Legend)")
        ax.set_ylim(0, 1.1)
        
        handles, labels = ax.get_legend_handles_labels()
        leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.add_artist(leg_personas)
        
        ax.text(1.05, 0.7, f"Model Index Key:\n{global_legend_text}", transform=ax.transAxes, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                fontsize=9, verticalalignment='top')
        
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=3, fontweight='bold', fontsize=9)
            
        plt.tight_layout()
        fig.savefig(global_img_path / f"global_accuracy_reasoning_{reasoning_val}_{shot_val}.png", dpi=300, bbox_inches='tight')
        # plt.show()
        plt.close()
        
        # Grafico Globale Confidence Score
        fig, ax = plt.subplots(figsize=(10, 6))
        
        df_global_agg = df_global_config.groupby(['model_idx', 'personas'], as_index=False)['avg_confidence_logprob'].mean()
        df_global_agg['abs_confidence_logprob'] = df_global_agg['avg_confidence_logprob'].abs()
        df_global_agg = df_global_agg.sort_values('model_idx')
        
        barplot_global_conf = sns.barplot(
            data=df_global_agg,
            x='model_idx',
            y='abs_confidence_logprob',
            hue='personas',
            ax=ax,
            palette=personas_palette
        )
        
        ticks = ax.get_yticks()
        ax.set_yticks(ticks)
        ax.set_yticklabels([f"-{t:.2f}" if t != 0 else "0.00" for t in ticks])
        
        ax.set_title(f"GLOBAL ANALYSIS | Reasoning: {reasoning_val} | Shot: {shot_val}\nEffect of Personas on Extracted Confidence Score")
        ax.set_ylabel("Confidence Logprob")
        ax.set_xlabel("Models (See Legend)")
        
        handles, labels = ax.get_legend_handles_labels()
        leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.add_artist(leg_personas)
        
        ax.text(1.05, 0.7, f"Model Index Key:\n{global_legend_text}", transform=ax.transAxes, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                fontsize=9, verticalalignment='top')
        
        for p in barplot_global_conf.patches:
            height = p.get_height()
            if not np.isnan(height) and height > 0:
                original_val = -height
                ax.annotate(f'{original_val:.4f}',
                            (p.get_x() + p.get_width() / 2., height),
                            ha='center', va='bottom',
                            xytext=(0, 3), textcoords='offset points', fontweight='bold', fontsize=8)
        
        plt.tight_layout()
        fig.savefig(global_img_path / f"global_confidence_reasoning_{reasoning_val}_{shot_val}.png", dpi=300, bbox_inches='tight')
        # plt.show()
        plt.close()

## 2. Intra-Model Analysis

This section explores how parameter modifications affect the stability, performance, and confidence metrics of each model individually.

### Generating Intra-Model Reports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

if not df_metrics.empty:
    images_base_path = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/images")
    intra_images_path = images_base_path / "intra_model"
    intra_images_path.mkdir(parents=True, exist_ok=True)
    
    sns.set_theme(style="whitegrid")
    personas_palette = {False: "#7f8c8d", True: "#e67e22"}
    
    print("==================================================")
    print("### 2.1 INTRA-MODEL PARAMETER IMPACT ANALYSIS")
    print("==================================================")
    
    unique_models = df_metrics['model'].unique()
    
    for model_name in unique_models:
        print(f"\nProcessing Intra-Model Plots for: {model_name}")
        df_model = df_metrics[df_metrics['model'] == model_name].copy()
        
        model_img_path = intra_images_path / model_name
        model_img_path.mkdir(parents=True, exist_ok=True)
        
        # Riassunto Pivot originale
        intra_model_pivot = df_model.pivot_table(
            index=['dataset'],
            columns=['personas', 'reasoning', 'shot'],
            values=['avg_mean_accuracy', 'avg_confidence_logprob']
        )
        print(f"Pivot Table Summary for {model_name}:")
        display(intra_model_pivot.style.background_gradient(cmap='Oranges', axis=1))
        
        for dataset_name, df_model_dataset in df_model.groupby('dataset'):
            df_model_dataset = df_model_dataset.copy()
            
            # Generiamo l'elenco univoco delle sotto-configurazioni sperimentali stabili (senza personas)
            # Applicando l'ordinamento richiesto: prima per 'reasoning' e poi per 'shot'
            sub_configs = df_model_dataset[['reasoning', 'shot']].drop_duplicates()
            sub_configs = sub_configs.sort_values(by=['reasoning', 'shot']).reset_index(drop=True)
            
            # Creiamo la mappatura numerica indicizzata (Setup 1, Setup 2...)
            config_to_idx = {}
            legend_lines = []
            for i, row in sub_configs.iterrows():
                idx_label = f"Setup {i+1}"
                config_to_idx[(row['reasoning'], row['shot'])] = idx_label
                legend_lines.append(f"{idx_label} -> Reasoning: {row['reasoning']} | Shot: {row['shot']}")
            
            config_legend_text = "\n".join(legend_lines)
            
            # Applichiamo l'indice numerico strutturato al dataframe corrente
            df_model_dataset['sub_config_idx'] = df_model_dataset.apply(
                lambda r: config_to_idx[(r['reasoning'], r['shot'])], axis=1
            )
            # Ordiniamo l'intero set per riflettere l'ordine numerico degli indici sull'asse X
            df_model_dataset = df_model_dataset.sort_values(by=['sub_config_idx', 'personas'])
            
            # ----------------------------------------------------
            # GRAFICO 1: INTRA-MODEL ACCURACY
            # ----------------------------------------------------
            fig, ax = plt.subplots(figsize=(12, 6))
            
            barplot_acc = sns.barplot(
                data=df_model_dataset,
                x='sub_config_idx',
                y='avg_mean_accuracy',
                hue='personas',
                ax=ax,
                palette=personas_palette
            )
            
            ax.set_title(f"Model: {model_name} | Dataset: {dataset_name}\nImpact of Personas on Accuracy Across Configurations")
            ax.set_ylabel("Accuracy")
            ax.set_xlabel("Experimental Setup Configurations (See Legend)")
            ax.set_ylim(0, 1.1)
            
            # Gestione Legende separate
            handles, labels = ax.get_legend_handles_labels()
            leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.add_artist(leg_personas)
            
            ax.text(1.05, 0.7, f"Configuration Key:\n{config_legend_text}", transform=ax.transAxes, 
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                    fontsize=9, verticalalignment='top')
            
            for container in ax.containers:
                ax.bar_label(container, fmt='%.3f', padding=3, fontweight='bold', fontsize=9)
                
            plt.tight_layout()
            filename_acc = f"{dataset_name}_intra_accuracy.png"
            fig.savefig(model_img_path / filename_acc, dpi=300, bbox_inches='tight')
            # plt.show()
            plt.close()
            
            # ----------------------------------------------------
            # GRAFICO 2: INTRA-MODEL CONFIDENCE (Valori Negativi Invertiti)
            # ----------------------------------------------------
            fig, ax = plt.subplots(figsize=(12, 6))
            
            df_model_dataset_inverted = df_model_dataset.copy()
            df_model_dataset_inverted['abs_confidence_logprob'] = df_model_dataset_inverted['avg_confidence_logprob'].abs()
            
            barplot_conf = sns.barplot(
                data=df_model_dataset_inverted,
                x='sub_config_idx',
                y='abs_confidence_logprob',
                hue='personas',
                ax=ax,
                palette=personas_palette
            )
            
            ticks = ax.get_yticks()
            ax.set_yticks(ticks)
            ax.set_yticklabels([f"-{t:.2f}" if t != 0 else "0.00" for t in ticks])
            
            ax.set_title(f"Model: {model_name} | Dataset: {dataset_name}\nImpact of Personas on Extracted Confidence Score")
            ax.set_ylabel("Confidence Logprob")
            ax.set_xlabel("Experimental Setup Configurations (See Legend)")
            
            handles, labels = ax.get_legend_handles_labels()
            leg_personas = ax.legend(handles, ["False", "True"], title="Personas Effect", bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.add_artist(leg_personas)
            
            ax.text(1.05, 0.7, f"Configuration Key:\n{config_legend_text}", transform=ax.transAxes, 
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#000000", alpha=0.1),
                    fontsize=9, verticalalignment='top')
            
            for p in barplot_conf.patches:
                height = p.get_height()
                if not np.isnan(height) and height > 0:
                    original_val = -height
                    ax.annotate(f'{original_val:.4f}',
                                (p.get_x() + p.get_width() / 2., height),
                                ha='center', va='bottom',
                                xytext=(0, 3), textcoords='offset points', fontweight='bold', fontsize=8)
            
            plt.tight_layout()
            filename_conf = f"{dataset_name}_intra_confidence.png"
            fig.savefig(model_img_path / filename_conf, dpi=300, bbox_inches='tight')
            # plt.show()
            plt.close()
            
        print("\n" + "-"*40 + "\n")
else:
    print("The metrics dataframe is empty. Please check the data loading step.")